# Laboratorio: Diseñando Combos para "Café COVA" ☕

**Contexto:**
La gerencia de la cafetería de la facultad quiere aumentar sus ingresos. Saben que los estudiantes compran cosas en grupo, pero no saben exactamente qué. Tu trabajo como Ingeniero de Datos es leer los registros crudos de la caja registradora, transformar los datos a una matriz transaccional y aplicar el algoritmo Apriori para encontrar los "Combos" más rentables.

In [12]:
pip install mlxtend

Note: you may need to restart the kernel to use updated packages.


In [13]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# FASE 1: EXTRACCIÓN (Datos de producción)
# El sistema punto de venta (POS) exporta los tickets como una lista de listas.
tickets_cafeteria = [
    ['Cafe Americano', 'Croissant', 'Agua'],
    ['Frappe Mocha', 'Muffin', 'Agua'],
    ['Cafe Americano', 'Croissant', 'Jugo de Naranja'],
    ['Te Verde', 'Galletas', 'Agua'],
    ['Frappe Mocha', 'Muffin', 'Galletas'],
    ['Cafe Americano', 'Croissant', 'Agua', 'Galletas'],
    ['Frappe Mocha', 'Brownie', 'Agua'],
    ['Te Verde', 'Ensalada', 'Agua'],
    ['Cafe Americano', 'Brownie', 'Jugo de Naranja'],
    ['Frappe Mocha', 'Muffin', 'Brownie']
]

### Reto 1: Ingeniería de Características (De Lista a Matriz)
El algoritmo Apriori no entiende listas de texto. Necesitas convertir `tickets_cafeteria` en un DataFrame de Pandas donde cada columna sea un producto y cada fila (ticket) tenga un `True` o `False` (One-Hot Encoding transaccional).

In [14]:
# TU CÓDIGO AQUÍ:
h = TransactionEncoder()
matriz = h.fit_transform(tickets_cafeteria)
df_cafe_bool = pd.DataFrame(matriz, columns=h.columns_)
# Utiliza TransactionEncoder para crear df_cafe_bool

### Reto 2: Extracción de Patrones Frecuentes
La gerencia te pide descartar las coincidencias raras. Extrae únicamente los `itemsets` (combinaciones de productos) que aparezcan en **al menos el 25% (0.25)** de los tickets.

In [15]:
# TU CÓDIGO AQUÍ:
# Usa la función apriori() pasándole tu df_cafe_bool. Guarda el resultado en itemsets_cafe
itemsets = apriori(df_cafe_bool, min_support=0.25, use_colnames=True)

### Reto 3: Creación de Reglas de Asociación
A partir de los itemsets frecuentes, genera las reglas de asociación. Filtra el resultado para mostrar **únicamente** aquellas reglas que tengan una `Confianza` superior al 60% (0.6).

In [16]:
# TU CÓDIGO AQUÍ:
# Usa association_rules() y guarda el resultado en reglas_cafe
# Filtra las columnas para imprimir solo: antecedents, consequents, support, confidence, lift
reglas = association_rules(itemsets, metric="confidence", min_threshold=0.6)
display(reglas[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

,antecedents,consequents,support,confidence,lift
0,(Cafe Americano),(Croissant),0.3,0.75,2.5
1,(Croissant),(Cafe Americano),0.3,1.00,2.5
2,(Frappe Mocha),(Muffin),0.3,0.75,2.5
3,(Muffin),(Frappe Mocha),0.3,1.00,2.5


### 🧠 Reto 4: Defensa de Ingeniería (Análisis de Negocio)

Revisa la tabla resultante de tus reglas y responde las siguientes preguntas para el gerente de la cafetería:

1. **La regla de mayor Confianza:** Identifica la regla que tiene el mayor nivel de Confianza. Traduce esa regla al español simple (ej. "Si un alumno compra X, existe un Y% de probabilidad de que también compre Z").
2. **El engaño del Soporte vs Lift:** ¿Qué producto de la cafetería se vende muchísimo (alto soporte), pero no deberíamos ponerlo en promoción (combo) con el Frappe Mocha porque su Lift es bajo o cercano a 1? Explica matemáticamente por qué.

1. Las reglas con mayor nivel de confianza son las de croissant-cafe americano y muffin-frappe. 
Si un alumno compra un croissant hay 100% de probabilidad de que también compre un café americano. 
De igual manera, si un alumno compra un muffin hay un 100% de probabilidad de que también compre un muffin.

2. Sería el agua porque se vende en casi todos los tickets (alto soporte). Que exista la coincidencia que al comprar un frappe se compre tambien un agua no quiere decir que su venta se base en esa razon. El agua se vende bien por si sola y un lift con ese resultado indica que no es buena idea hacer el combo entre esos dos productos. 